Logistic Regression on Spanish EIT Human Scores

In [ ]:

import pandas as pd

workbooks = [data, data2]
records = []

for wb in workbooks:
    for ws in wb.worksheets:
        rows = list(ws.iter_rows(values_only=True))
        if not rows:
            continue

        headers = [str(h).strip().lower() if h is not None else "" for h in rows[0]]

        if "stimulus" not in headers or "final transcription" not in headers or "final score" not in headers:
            continue

        stim_idx = headers.index("stimulus")
        trans_idx = headers.index("final transcription")
        score_idx = headers.index("final score")

        for row in rows[1:]:
            stim = row[stim_idx] if stim_idx < len(row) else None
            trans = row[trans_idx] if trans_idx < len(row) else None
            score = row[score_idx] if score_idx < len(row) else None

            if score is None or (isinstance(score, str) and score.strip() == ""):
                continue

            records.append({
                "stimulus": stim,
                "final transcription": trans,
                "final score": score,
            })

df = pd.DataFrame(records, columns=["stimulus", "final transcription", "final score"])
df.to_excel("combined_stimulus_final_transcription_scores.xlsx", index=False)

print(f"Saved {len(df)} rows to combined_stimulus_final_transcription_scores.xlsx")

Saved 2043 rows to combined_stimulus_final_transcription_scores.xlsx


In [ ]:
#removing rows with null stimulus

df = df[df["stimulus"].notna()].copy()

df.to_excel("combined_stimulus_final_transcription_scores.xlsx", index=False)

print(f"Saved {len(df)} rows to combined.xlsx")

Saved 1974 rows to combined_stimulus_final_transcription_scores.xlsx


In [7]:
# Clean the data: convert final score to numeric and remove rows with formulas or non-numeric values

# Convert final score to numeric, coercing errors to NaN
df["final score"] = pd.to_numeric(df["final score"], errors='coerce')

# Remove rows where final score is NaN (couldn't be converted)
df = df.dropna(subset=["final score"]).copy()

print(f"Data shape after cleaning: {df.shape}")
print(f"\nFinal Score Statistics:")
print(df["final score"].describe())
print(f"\nData types:")
print(df.dtypes)

Data shape after cleaning: (1964, 3)

Final Score Statistics:
count    1964.000000
mean        1.821792
std         1.290592
min         0.000000
25%         1.000000
50%         2.000000
75%         3.000000
max         4.000000
Name: final score, dtype: float64

Data types:
stimulus                object
final transcription     object
final score            float64
dtype: object


In [9]:
#creating a test set, train set, and validation set

from sklearn.model_selection import train_test_split

# Ensure we're using clean numeric data for the split
df_clean = df[["stimulus", "final transcription", "final score"]].copy()
df_clean["final score"] = pd.to_numeric(df_clean["final score"], errors='coerce')
df_clean = df_clean.dropna()

train_df, test_df = train_test_split(df_clean, test_size=0.2, random_state=42)
train_df, val_df = train_test_split(train_df, test_size=0.25, random_state=42)  # 0.25 * 0.8 = 0.2

print(f"Train set: {train_df.shape[0]} samples")
print(f"Validation set: {val_df.shape[0]} samples")
print(f"Test set: {test_df.shape[0]} samples")

Train set: 1178 samples
Validation set: 393 samples
Test set: 393 samples


In [13]:
# Feature extraction using TF-IDF on the transcription text

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
import numpy as np

# Create TF-IDF vectorizer
vectorizer = TfidfVectorizer(max_features=500, lowercase=True, stop_words='english')

# Fit on training data
X_train = vectorizer.fit_transform(train_df["final transcription"].astype(str))
X_val = vectorizer.transform(val_df["final transcription"].astype(str))
X_test = vectorizer.transform(test_df["final transcription"].astype(str))

# Prepare target variable (convert score to binary classification)
# Assuming score needs to be converted to binary (e.g., pass/fail or high/low)
y_train = (train_df["final score"].astype(float) >= train_df["final score"].astype(float).median()).astype(int)
y_val = (val_df["final score"].astype(float) >= train_df["final score"].astype(float).median()).astype(int)
y_test = (test_df["final score"].astype(float) >= train_df["final score"].astype(float).median()).astype(int)

print(f"Training set size: {X_train.shape[0]}")
print(f"Validation set size: {X_val.shape[0]}")
print(f"Test set size: {X_test.shape[0]}")
print(f"Feature dimension: {X_train.shape[1]}")
print(f"\nClass distribution (train): {np.bincount(y_train)}")


Training set size: 1178
Validation set size: 393
Test set size: 393
Feature dimension: 500

Class distribution (train): [514 664]


In [14]:
# Train logistic regression model

lr_model = LogisticRegression(max_iter=1000, random_state=42, solver='lbfgs')
lr_model.fit(X_train, y_train)

# Make predictions
y_train_pred = lr_model.predict(X_train)
y_val_pred = lr_model.predict(X_val)
y_test_pred = lr_model.predict(X_test)

# Evaluate on all sets
print("=" * 60)
print("TRAINING SET PERFORMANCE")
print("=" * 60)
print(f"Accuracy:  {accuracy_score(y_train, y_train_pred):.4f}")
print(f"Precision: {precision_score(y_train, y_train_pred):.4f}")
print(f"Recall:    {recall_score(y_train, y_train_pred):.4f}")
print(f"F1 Score:  {f1_score(y_train, y_train_pred):.4f}")

print("\n" + "=" * 60)
print("VALIDATION SET PERFORMANCE")
print("=" * 60)
print(f"Accuracy:  {accuracy_score(y_val, y_val_pred):.4f}")
print(f"Precision: {precision_score(y_val, y_val_pred):.4f}")
print(f"Recall:    {recall_score(y_val, y_val_pred):.4f}")
print(f"F1 Score:  {f1_score(y_val, y_val_pred):.4f}")

print("\n" + "=" * 60)
print("TEST SET PERFORMANCE")
print("=" * 60)
print(f"Accuracy:  {accuracy_score(y_test, y_test_pred):.4f}")
print(f"Precision: {precision_score(y_test, y_test_pred):.4f}")
print(f"Recall:    {recall_score(y_test, y_test_pred):.4f}")
print(f"F1 Score:  {f1_score(y_test, y_test_pred):.4f}")

print("\n" + "=" * 60)
print("CONFUSION MATRIX (Test Set)")
print("=" * 60)
cm = confusion_matrix(y_test, y_test_pred)
print(cm)

print("\n" + "=" * 60)
print("CLASSIFICATION REPORT (Test Set)")
print("=" * 60)
print(classification_report(y_test, y_test_pred))


TRAINING SET PERFORMANCE
Accuracy:  0.8625
Precision: 0.8410
Recall:    0.9322
F1 Score:  0.8843

VALIDATION SET PERFORMANCE
Accuracy:  0.7634
Precision: 0.7276
Recall:    0.8732
F1 Score:  0.7938

TEST SET PERFORMANCE
Accuracy:  0.7608
Precision: 0.7610
Recall:    0.8489
F1 Score:  0.8025

CONFUSION MATRIX (Test Set)
[[108  60]
 [ 34 191]]

CLASSIFICATION REPORT (Test Set)
              precision    recall  f1-score   support

           0       0.76      0.64      0.70       168
           1       0.76      0.85      0.80       225

    accuracy                           0.76       393
   macro avg       0.76      0.75      0.75       393
weighted avg       0.76      0.76      0.76       393



SVM

In [15]:
# Train SVM model for comparison

from sklearn.svm import SVC

svm_model = SVC(kernel='rbf', C=1.0, gamma='scale', random_state=42)
svm_model.fit(X_train, y_train)

# Make predictions
svm_train_pred = svm_model.predict(X_train)
svm_val_pred = svm_model.predict(X_val)
svm_test_pred = svm_model.predict(X_test)

# Evaluate on all sets
print("=" * 60)
print("SVM - TRAINING SET PERFORMANCE")
print("=" * 60)
print(f"Accuracy:  {accuracy_score(y_train, svm_train_pred):.4f}")
print(f"Precision: {precision_score(y_train, svm_train_pred):.4f}")
print(f"Recall:    {recall_score(y_train, svm_train_pred):.4f}")
print(f"F1 Score:  {f1_score(y_train, svm_train_pred):.4f}")

print("\n" + "=" * 60)
print("SVM - VALIDATION SET PERFORMANCE")
print("=" * 60)
print(f"Accuracy:  {accuracy_score(y_val, svm_val_pred):.4f}")
print(f"Precision: {precision_score(y_val, svm_val_pred):.4f}")
print(f"Recall:    {recall_score(y_val, svm_val_pred):.4f}")
print(f"F1 Score:  {f1_score(y_val, svm_val_pred):.4f}")

print("\n" + "=" * 60)
print("SVM - TEST SET PERFORMANCE")
print("=" * 60)
print(f"Accuracy:  {accuracy_score(y_test, svm_test_pred):.4f}")
print(f"Precision: {precision_score(y_test, svm_test_pred):.4f}")
print(f"Recall:    {recall_score(y_test, svm_test_pred):.4f}")
print(f"F1 Score:  {f1_score(y_test, svm_test_pred):.4f}")

print("\n" + "=" * 60)
print("CONFUSION MATRIX (Test Set)")
print("=" * 60)
svm_cm = confusion_matrix(y_test, svm_test_pred)
print(svm_cm)

print("\n" + "=" * 60)
print("CLASSIFICATION REPORT (Test Set)")
print("=" * 60)
print(classification_report(y_test, svm_test_pred))

SVM - TRAINING SET PERFORMANCE
Accuracy:  0.9576
Precision: 0.9515
Recall:    0.9744
F1 Score:  0.9628

SVM - VALIDATION SET PERFORMANCE
Accuracy:  0.8397
Precision: 0.8114
Recall:    0.9024
F1 Score:  0.8545

SVM - TEST SET PERFORMANCE
Accuracy:  0.8193
Precision: 0.8438
Recall:    0.8400
F1 Score:  0.8419

CONFUSION MATRIX (Test Set)
[[133  35]
 [ 36 189]]

CLASSIFICATION REPORT (Test Set)
              precision    recall  f1-score   support

           0       0.79      0.79      0.79       168
           1       0.84      0.84      0.84       225

    accuracy                           0.82       393
   macro avg       0.82      0.82      0.82       393
weighted avg       0.82      0.82      0.82       393



In [ ]:
# SVM Hyperparameter Tuning with GridSearchCV

from sklearn.model_selection import GridSearchCV

param_grid = {
    'C': [0.1, 1, 10, 100],
    'gamma': ['scale', 'auto', 0.001, 0.01, 0.1, 1]
}

svm_grid = SVC(kernel='rbf', random_state=42)
grid_search = GridSearchCV(svm_grid, param_grid, cv=5, scoring='f1', n_jobs=-1, verbose=1)

print("Training SVM with GridSearchCV (5-fold cross-validation)...")
grid_search.fit(X_train, y_train)

print(f"\nBest parameters: {grid_search.best_params_}")
print(f"Best CV F1 Score: {grid_search.best_score_:.4f}")

# Train final model with best parameters
best_svm = grid_search.best_estimator_

# Make predictions with optimized SVM
best_svm_train_pred = best_svm.predict(X_train)
best_svm_val_pred = best_svm.predict(X_val)
best_svm_test_pred = best_svm.predict(X_test)

# Evaluate optimized SVM
print("\n" + "=" * 60)
print("OPTIMIZED SVM - TEST SET PERFORMANCE")
print("=" * 60)
print(f"Accuracy:  {accuracy_score(y_test, best_svm_test_pred):.4f}")
print(f"Precision: {precision_score(y_test, best_svm_test_pred):.4f}")
print(f"Recall:    {recall_score(y_test, best_svm_test_pred):.4f}")
print(f"F1 Score:  {f1_score(y_test, best_svm_test_pred):.4f}")

print("\n" + "=" * 60)
print("CONFUSION MATRIX")
print("=" * 60)
best_svm_cm = confusion_matrix(y_test, best_svm_test_pred)
print(best_svm_cm)

print("\n" + "=" * 60)
print("CLASSIFICATION REPORT")
print("=" * 60)
print(classification_report(y_test, best_svm_test_pred))

# Compare all three models
print("\n" + "=" * 60)
print("MODEL COMPARISON (Test Set)")
print("=" * 60)
print(f"Logistic Regression: Accuracy={accuracy_score(y_test, y_test_pred):.4f}, F1={f1_score(y_test, y_test_pred):.4f}")
print(f"SVM (default):       Accuracy={accuracy_score(y_test, svm_test_pred):.4f}, F1={f1_score(y_test, svm_test_pred):.4f}")
print(f"SVM (optimized):     Accuracy={accuracy_score(y_test, best_svm_test_pred):.4f}, F1={f1_score(y_test, best_svm_test_pred):.4f}")

Training SVM with GridSearchCV (5-fold cross-validation)...
Fitting 5 folds for each of 24 candidates, totalling 120 fits

Best parameters: {'C': 10, 'gamma': 'scale'}
Best CV F1 Score: 0.8329

OPTIMIZED SVM - TEST SET PERFORMANCE
Accuracy:  0.8219
Precision: 0.8539
Recall:    0.8311
F1 Score:  0.8423

CONFUSION MATRIX
[[136  32]
 [ 38 187]]

CLASSIFICATION REPORT
              precision    recall  f1-score   support

           0       0.78      0.81      0.80       168
           1       0.85      0.83      0.84       225

    accuracy                           0.82       393
   macro avg       0.82      0.82      0.82       393
weighted avg       0.82      0.82      0.82       393


MODEL COMPARISON (Test Set)
Logistic Regression: Accuracy=0.7608, F1=0.8025
SVM (default):       Accuracy=0.8193, F1=0.8419
SVM (optimized):     Accuracy=0.8219, F1=0.8423


In [17]:
# NLP: Sentence embeddings using Sentence-BERT

from sentence_transformers import SentenceTransformer
import numpy as np

# Load pre-trained sentence transformer model
print("Loading Sentence-BERT model...")
model = SentenceTransformer('all-MiniLM-L6-v2')  # Lightweight, fast model

# Generate embeddings for transcriptions
print("Generating embeddings for training set...")
X_train_embeddings = model.encode(train_df["final transcription"].astype(str).tolist(), show_progress_bar=True)

print("Generating embeddings for validation set...")
X_val_embeddings = model.encode(val_df["final transcription"].astype(str).tolist(), show_progress_bar=True)

print("Generating embeddings for test set...")
X_test_embeddings = model.encode(test_df["final transcription"].astype(str).tolist(), show_progress_bar=True)

print(f"\nEmbedding shape: {X_train_embeddings.shape}")

# Train SVM with embeddings (using best parameters from grid search)
svm_embeddings = SVC(kernel='rbf', C=1.0, gamma='scale', random_state=42)
svm_embeddings.fit(X_train_embeddings, y_train)

# Make predictions
svm_emb_train_pred = svm_embeddings.predict(X_train_embeddings)
svm_emb_val_pred = svm_embeddings.predict(X_val_embeddings)
svm_emb_test_pred = svm_embeddings.predict(X_test_embeddings)

# Evaluate
print("\n" + "=" * 60)
print("SVM + SENTENCE-BERT EMBEDDINGS - TEST SET")
print("=" * 60)
print(f"Accuracy:  {accuracy_score(y_test, svm_emb_test_pred):.4f}")
print(f"Precision: {precision_score(y_test, svm_emb_test_pred):.4f}")
print(f"Recall:    {recall_score(y_test, svm_emb_test_pred):.4f}")
print(f"F1 Score:  {f1_score(y_test, svm_emb_test_pred):.4f}")

print("\n" + "=" * 60)
print("FINAL MODEL COMPARISON (Test Set)")
print("=" * 60)
print(f"Logistic Regression (TF-IDF):    Accuracy={accuracy_score(y_test, y_test_pred):.4f}, F1={f1_score(y_test, y_test_pred):.4f}")
print(f"SVM (TF-IDF):                    Accuracy={accuracy_score(y_test, svm_test_pred):.4f}, F1={f1_score(y_test, svm_test_pred):.4f}")
print(f"SVM (Sentence-BERT):             Accuracy={accuracy_score(y_test, svm_emb_test_pred):.4f}, F1={f1_score(y_test, svm_emb_test_pred):.4f}")

print("\n" + "=" * 60)
print("CONFUSION MATRIX - Sentence-BERT")
print("=" * 60)
print(confusion_matrix(y_test, svm_emb_test_pred))

print("\n" + "=" * 60)
print("CLASSIFICATION REPORT - Sentence-BERT")
print("=" * 60)
print(classification_report(y_test, svm_emb_test_pred))

C:\Users\khann\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading Sentence-BERT model...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 240.29it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Generating embeddings for training set...


Batches: 100%|██████████| 37/37 [00:07<00:00,  5.06it/s]


Generating embeddings for validation set...


Batches: 100%|██████████| 13/13 [00:02<00:00,  6.06it/s]


Generating embeddings for test set...


Batches: 100%|██████████| 13/13 [00:01<00:00,  8.25it/s]



Embedding shape: (1178, 384)

SVM + SENTENCE-BERT EMBEDDINGS - TEST SET
Accuracy:  0.7990
Precision: 0.7992
Recall:    0.8667
F1 Score:  0.8316

FINAL MODEL COMPARISON (Test Set)
Logistic Regression (TF-IDF):    Accuracy=0.7608, F1=0.8025
SVM (TF-IDF):                    Accuracy=0.8193, F1=0.8419
SVM (Sentence-BERT):             Accuracy=0.7990, F1=0.8316

CONFUSION MATRIX - Sentence-BERT
[[119  49]
 [ 30 195]]

CLASSIFICATION REPORT - Sentence-BERT
              precision    recall  f1-score   support

           0       0.80      0.71      0.75       168
           1       0.80      0.87      0.83       225

    accuracy                           0.80       393
   macro avg       0.80      0.79      0.79       393
weighted avg       0.80      0.80      0.80       393

